# Research Agent RL — Week 3 (GRPO): End-to-End LLM Training

**Setup:** GPU T4 x1, Internet ON

This notebook trains the **LLM itself** (not just the MLP stopper) using GRPO,
the same algorithm behind DeepSeek-R1. Unsloth cuts VRAM by ~80%, making this
feasible on a free T4.

**Key difference from `kaggle_week3_rl.ipynb`:**
- Week 3 RL: LLM frozen, tiny MLP learns when to stop (REINFORCE)
- **This notebook**: LLM itself learns to reason + stop efficiently (GRPO)

**How GRPO works here:**
1. For each HotpotQA question, sample G=4 full reasoning traces from the LLM
2. Score each trace: correctness + format quality + step efficiency
3. Compute within-group advantages (no critic network needed)
4. Update LLM weights to favour higher-reward traces

The LLM generates the full trace in one shot (including 'virtual' search
observations from parametric memory). No live tool execution during training.

## 1. Check GPU

In [ ]:
import subprocess
r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(r.stdout if r.returncode == 0 else 'No GPU — enable T4 in Settings!')

## 2. Clone repo & install Unsloth

In [ ]:
import os

REPO_URL = 'https://github.com/202520030411/Research_Agent_RL.git'
REPO_DIR = '/kaggle/working/Research_Agent_RL'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)
print('Working directory:', os.getcwd())

In [ ]:
# Install Unsloth (optimised GRPO kernels, 80% less VRAM)
!pip install -q unsloth vllm
# Install remaining project dependencies
!pip install -q datasets pyyaml tqdm

## 3. Load Qwen2.5-0.5B with Unsloth (4-bit QLoRA)

In [ ]:
import torch
from unsloth import FastLanguageModel

MODEL_NAME     = 'Qwen/Qwen2.5-0.5B-Instruct'
MAX_SEQ_LENGTH = 1024   # full trace fits in 1K tokens for 0.5B
LORA_RANK      = 16

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,          # QLoRA — ~0.5GB on T4
    fast_inference=True,        # Unsloth vLLM-based fast generation
    max_lora_rank=LORA_RANK,
    gpu_memory_utilization=0.6, # leave headroom for GRPO logits
)

# Wrap with LoRA adapters (only these ~4M params will be trained)
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=LORA_RANK,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)

print(f'VRAM after model load: {torch.cuda.memory_allocated()/1e9:.2f} GB')

## 4. Load HotpotQA and build GRPO dataset

In [ ]:
from datasets import load_dataset, Dataset
from data.sft_dataset import SYSTEM_PROMPT

N_TRAIN = 2000   # training questions
N_VAL   = 200    # validation questions

print('Loading HotpotQA...')
hf       = load_dataset('hotpot_qa', 'distractor', trust_remote_code=True)
train_hf = hf['train'].shuffle(seed=42)
val_hf   = hf['validation']

def make_prompt(question: str) -> str:
    """Build the chat-formatted prompt for one question."""
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': f'Question: {question}'},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

# GRPOTrainer expects columns: 'prompt' + any extra columns passed to reward fns
train_data = Dataset.from_dict({
    'prompt': [make_prompt(train_hf[i]['question']) for i in range(N_TRAIN)],
    'answer': [train_hf[i]['answer']               for i in range(N_TRAIN)],
})
val_data = Dataset.from_dict({
    'prompt': [make_prompt(val_hf[i]['question']) for i in range(N_VAL)],
    'answer': [val_hf[i]['answer']               for i in range(N_VAL)],
})

print(f'Train: {len(train_data)} | Val: {len(val_data)}')
print('Example prompt (truncated):')
print(train_data[0]['prompt'][:300])

## 5. Define reward functions

In [ ]:
from rl.grpo_rewards import correctness_reward, format_reward, efficiency_reward

# Smoke-test reward functions
test_comp = [
    '{"thought": "Let me look this up.", "action": "search", "query": "test", "confidence": 0.4}\n'
    'Observation: Some result.\n'
    '{"thought": "I know the answer.", "action": "answer", "answer": "Paris", "confidence": 0.9}'
]
test_gold = ['Paris']

print('Correctness:', correctness_reward(test_comp, answer=test_gold))  # [1.0]
print('Format:     ', format_reward(test_comp))                          # [0.3]
print('Efficiency: ', efficiency_reward(test_comp))                      # negative

## 6. Configure GRPOTrainer

In [ ]:
from trl import GRPOTrainer, GRPOConfig

training_args = GRPOConfig(
    # --- Output ---
    output_dir='checkpoints/grpo-qwen',

    # --- Scale: tuned for T4 16GB ---
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,   # effective batch = 4 prompts
    num_generations=4,               # G=4 completions per prompt (GRPO group size)
    max_prompt_length=256,
    max_completion_length=512,       # full trace in ≤512 tokens

    # --- Optimisation ---
    learning_rate=5e-6,
    num_train_epochs=1,
    warmup_ratio=0.05,
    lr_scheduler_type='cosine',
    optim='adamw_8bit',              # 8-bit Adam saves ~1GB
    fp16=True,                       # T4 doesn't support bf16

    # --- GRPO-specific ---
    use_vllm=True,                   # fast generation via Unsloth+vLLM
    vllm_gpu_memory_utilization=0.4, # balance with training memory
    temperature=0.9,                 # diversity in sampled completions
    epsilon=0.2,                     # PPO-style clip for stability

    # --- Logging & saving ---
    logging_steps=5,
    save_steps=100,
    save_total_limit=2,
    report_to='none',
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[
        correctness_reward,  # main signal: +1 for correct answer
        format_reward,       # +0.3 for well-structured trace
        efficiency_reward,   # step + JSD penalty
    ],
    args=training_args,
    train_dataset=train_data,
    eval_dataset=val_data,
)

print('Trainer ready.')
print(f'Steps per epoch: {len(trainer.get_train_dataloader())}')

## 7. Train

In [ ]:
import torch
torch.cuda.empty_cache()

print('Starting GRPO training...')
print('Watch for the reward column to increase over steps.\n')

trainer.train()

print('\nTraining complete!')
print(f'Peak VRAM: {torch.cuda.max_memory_allocated()/1e9:.2f} GB')

## 8. Save the fine-tuned model

In [ ]:
import os

GRPO_ADAPTER_DIR  = '/kaggle/working/qwen-grpo-adapter'
GRPO_MERGED_DIR   = '/kaggle/working/qwen-grpo-merged'

# Save LoRA adapter only (small, fast)
model.save_pretrained(GRPO_ADAPTER_DIR)
tokenizer.save_pretrained(GRPO_ADAPTER_DIR)
print(f'LoRA adapter saved → {GRPO_ADAPTER_DIR}')

# Optionally save merged 16-bit model
# model.save_pretrained_merged(GRPO_MERGED_DIR, tokenizer, save_method='merged_16bit')

!ls -lh {GRPO_ADAPTER_DIR}

## 9. Evaluate GRPO model vs SFT baseline

In [ ]:
# Reload model in inference mode (faster)
from unsloth import FastLanguageModel
import torch

torch.cuda.empty_cache()

eval_model, eval_tokenizer = FastLanguageModel.from_pretrained(
    model_name=GRPO_ADAPTER_DIR,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(eval_model)
print('Model loaded for inference.')

In [ ]:
from rl.grpo_rewards import parse_trace, _is_correct
from tqdm import tqdm

N_EVAL = 100

correct = 0
total_steps = 0
results = []

for i in tqdm(range(N_EVAL), desc='Evaluating GRPO model'):
    q    = val_hf[i]['question']
    gold = val_hf[i]['answer']

    prompt  = make_prompt(q)
    inputs  = eval_tokenizer(prompt, return_tensors='pt').to('cuda')
    out_ids = eval_model.generate(
        **inputs,
        max_new_tokens=400,
        temperature=0.1,
        do_sample=True,
        pad_token_id=eval_tokenizer.eos_token_id,
    )
    completion = eval_tokenizer.decode(
        out_ids[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True
    )
    trace = parse_trace(completion)
    ok    = _is_correct(trace['final_answer'], gold)

    correct     += int(ok)
    total_steps += trace['n_steps']
    results.append({
        'question': q, 'gold': gold,
        'pred': trace['final_answer'],
        'correct': ok, 'n_steps': trace['n_steps'],
    })

acc       = correct / N_EVAL
avg_steps = total_steps / N_EVAL
print(f'\nGRPO model — Accuracy: {acc:.3f}  Avg steps: {avg_steps:.1f}  Efficiency: {acc/max(avg_steps,1):.4f}')

## 10. Save evaluation results

In [ ]:
import json

# Save eval results
eval_path = '/kaggle/working/grpo_eval_results.json'
with open(eval_path, 'w') as f:
    json.dump({
        'summary': {
            'accuracy':   acc,
            'avg_steps':  avg_steps,
            'efficiency': acc / max(avg_steps, 1),
            'n_eval':     N_EVAL,
        },
        'results': results,
    }, f, indent=2)
print(f'Results saved → {eval_path}')

# Save training log
log_path = '/kaggle/working/grpo_training_log.json'
with open(log_path, 'w') as f:
    json.dump(trainer.state.log_history, f, indent=2)
print(f'Training log saved → {log_path}')

print('\nAll outputs in the Kaggle Output tab.')